# Project: Recommendation System

> Part of the **Machine Learning Learning Series**

## Project Overview
**Goal**: Build a movie recommendation system.

**Skills demonstrated**: Collaborative filtering, content-based filtering, hybrid system.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

np.random.seed(42)

# Synthetic movies
movies = pd.DataFrame({
    'movie_id': range(20),
    'title': [f'Movie {i}' for i in range(20)],
    'genres': np.random.choice(['action thriller', 'sci-fi drama', 'comedy romance', 'horror mystery', 'adventure fantasy'], 20)
})

# Synthetic ratings
ratings = pd.DataFrame({
    'user_id': np.random.randint(0, 50, 300),
    'movie_id': np.random.randint(0, 20, 300),
    'rating': np.random.choice([1,2,3,4,5], 300, p=[0.05,0.1,0.2,0.35,0.3])
}).drop_duplicates(['user_id','movie_id'])
print(f'Movies: {len(movies)}, Ratings: {len(ratings)}')

## Collaborative Filtering

In [ ]:
matrix = ratings.pivot_table(index='user_id', columns='movie_id', values='rating').fillna(0)
user_sim = cosine_similarity(matrix)

def cf_recommend(user_id, n=5):
    if user_id not in matrix.index: return []
    idx = list(matrix.index).index(user_id)
    similar_users = np.argsort(user_sim[idx])[::-1][1:6]
    scores = matrix.iloc[similar_users].mean(axis=0)
    rated = ratings[ratings['user_id']==user_id]['movie_id'].values
    unrated = [m for m in scores.index if m not in rated]
    return scores[unrated].nlargest(n).index.tolist()

recs = cf_recommend(0)
print('CF Recommendations for User 0:', [movies.loc[i,'title'] for i in recs])

## Content-Based Filtering

In [ ]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres'])
movie_sim = cosine_similarity(tfidf_matrix)

def cb_recommend(movie_id, n=5):
    scores = movie_sim[movie_id]
    similar = np.argsort(scores)[::-1][1:n+1]
    return similar.tolist()

recs = cb_recommend(0)
print('Content-Based Recommendations for Movie 0:', [movies.loc[i,'title'] for i in recs])